In [ ]:
import pandas as pd
import numpy as np

# =========================================
# Build full-sample OR analysis dataset (n=1,321)
# Non-actigraphy predictors: imputed on full sample
# Actigraphy predictors: left as real missing (n=575 available)
# =========================================

in_path = "../data/mcs_sh_sample_Feb_28.csv"
out_path = "../data/mcs_sh_sample_Feb_28_OR_preprocessed_FULLSAMPLE.csv"

df = pd.read_csv(in_path)
df.columns = df.columns.str.strip()

# Drop identifier and the eligibility criterion (constant in this subsample)
df = df.drop(columns=["id", "self_harm_14y"], errors="ignore")

# Structural missingness fix for cannabis (skip logic: never-users were
# never asked the frequency question, so missing = never used)
if "child_cannabis" in df.columns:
    df["child_cannabis"] = df["child_cannabis"].fillna("Never")

# Actigraphy columns: leave untouched (real missingness preserved)
actigraphy_cols = [
    "mean_acc_24h", "mean_acc_5to9", "m5_hour_start", "m5_mean_acc",
    "l5_hour_start", "l5_mean_acc", "mvpa_acc_5sec", "mvpa_acc_1min",
    "mvpa_acc_5min", "mvpa_bout1", "mvpa_bout5", "mvpa_bout10"
]
actigraphy_cols = [c for c in actigraphy_cols if c in df.columns]

target_col = "suicide_17y"
other_cols = [c for c in df.columns if c not in actigraphy_cols + [target_col]]

numeric_cols = df[other_cols].select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df[other_cols].select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# Median imputation for non-actigraphy numeric columns
df[numeric_cols] = df[numeric_cols].apply(lambda s: s.fillna(s.median()))

# Mode imputation for non-actigraphy categorical columns
for col in categorical_cols:
    mode_val = df[col].mode(dropna=True)
    fill_val = mode_val.iloc[0] if not mode_val.empty else "Unknown"
    df[col] = df[col].fillna(fill_val)

print("Remaining missing values (should only be actigraphy columns, ~746 each):")
print(df.isna().sum()[df.isna().sum() > 0])

df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Shape: {df.shape}")